In [1]:
pip install pandas numpy plotly pytz


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

from datetime import datetime
import pytz


In [3]:
apps = pd.read_csv("C:/Users/yashr/Desktop/eve/archive (5)/googleplaystore.csv")

reviews = pd.read_csv(
    "C:/Users/yashr/Desktop/eve/archive (5)/googleplaystore_user_reviews.csv"
)

In [4]:
print("Apps Dataset Shape:", apps.shape)
print("Reviews Dataset Shape:", reviews.shape)

Apps Dataset Shape: (10841, 13)
Reviews Dataset Shape: (64295, 5)


In [5]:
apps.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [6]:
reviews.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [7]:
#cleaning rating column
apps["Rating"] = pd.to_numeric(
    apps["Rating"],
    errors="coerce"
)

In [8]:
#cleaning reviews column
apps["Reviews"] = pd.to_numeric(
    apps["Reviews"],
    errors="coerce"
)

In [9]:
#cleaning installs columns
apps["Installs"] = (
    apps["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

apps["Installs"] = pd.to_numeric(
    apps["Installs"],
    errors="coerce"
)

In [10]:

#cleaning size column
def clean_size(size):

    if pd.isna(size):
        return np.nan

    size = str(size)

    if size == "Varies with device":
        return np.nan

    if size.endswith("M"):
        return float(size.replace("M", ""))

    if size.endswith("k"):
        return float(size.replace("k", "")) / 1024

    return np.nan

In [11]:
apps["Size_MB"] = apps["Size"].apply(
    clean_size
)

In [12]:
#Cleaning Sentiment Subjectivity
reviews["Sentiment_Subjectivity"] = (
    pd.to_numeric(
        reviews["Sentiment_Subjectivity"],
        errors="coerce"
    )
)

In [13]:
#Aggregating Sentiment per App
review_summary = (
    reviews
    .groupby("App", as_index=False)
    .agg(
        Avg_Subjectivity=(
            "Sentiment_Subjectivity",
            "mean"
        )
    )
)

In [14]:
#merging both the datasets

df = apps.merge(
    review_summary,
    on="App",
    how="inner"
)

In [15]:
#verifying merge
print(df.shape)
df.head()

(1532, 15)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Avg_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,14.0,0.641540
1,Garden Coloring Book,ART_AND_DESIGN,4.4,13791.0,33M,1000000.0,Free,0,Everyone,Art & Design,"September 20, 2017",2.9.2,3.0 and up,33.0,0.523447
2,3D Color Pixel by Number - Sandbox Art Coloring,ART_AND_DESIGN,4.4,1518.0,37M,100000.0,Free,0,Everyone,Art & Design,"August 3, 2018",1.2.3,2.3 and up,37.0,NaN
3,FlipaClip - Cartoon animation,ART_AND_DESIGN,4.3,194216.0,39M,5000000.0,Free,0,Everyone,Art & Design,"August 3, 2018",2.2.5,4.0.3 and up,39.0,0.679226
4,Boys Photo Editor - Six Pack & Men's Suit,ART_AND_DESIGN,4.1,654.0,12M,100000.0,Free,0,Everyone,Art & Design,"March 20, 2018",1.1,4.0.3 and up,12.0,0.479298


In [16]:
#defininig allowed categories
allowed_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

In [17]:
#Applying all filters
filtered_df = df[
    (df["Rating"] > 3.5)
    &
    (df["Reviews"] > 500)
    &
    (df["Installs"] > 50000)
    &
    (df["Avg_Subjectivity"] > 0.5)
    &
    (
        df["Category"]
        .isin(allowed_categories)
    )
    &
    (
        ~df["App"]
        .str.upper()
        .str.contains("S")
    )
].copy()

In [18]:
#translating categories
translation_map = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}

In [19]:
filtered_df["Display_Category"] = (
    filtered_df["Category"]
    .replace(translation_map)
)

In [20]:
#checking current IST time
ist = pytz.timezone(
    "Asia/Kolkata"
)

current_time = datetime.now(ist)

print(current_time)

2026-06-23 11:25:15.129410+05:30


In [21]:
#creating time rule
show_chart = (
    17 <= current_time.hour < 19
)

print(show_chart)

False


In [22]:
#creating color mapping
color_map = {
    "GAME": "pink"
}

In [23]:
#bubble chart
fig = px.scatter(
    filtered_df,

    x="Size_MB",

    y="Rating",

    size="Installs",

    color="Category",

    hover_name="App",

    color_discrete_map=color_map,

    title="App Size vs Rating"
)

In [24]:
import sys
!"{sys.executable}" -m pip install nbformat

In [25]:
import nbformat
print(nbformat.__version__)

5.10.4


In [28]:
fig.update_layout(
    xaxis_title="App Size (MB)",
    yaxis_title="Average Rating",
    legend_title="Category",
    template="plotly_white",
    height=600,
    width=1100
)

In [29]:
if show_chart:

    fig.show()

else:

    print(
        "Chart hidden."
        "\nVisible only between "
        "5 PM and 7 PM IST."
    )

Chart hidden.
Visible only between 5 PM and 7 PM IST.


In [30]:
print(
    filtered_df[
        [
            "App",
            "Category",
            "Rating",
            "Reviews",
            "Installs",
            "Avg_Subjectivity"
        ]
    ].head()
)

               App       Category  Rating   Reviews    Installs  \
55   Google Primer       BUSINESS     4.4   62272.0  10000000.0   
58    Call Blocker       BUSINESS     4.6  188841.0   5000000.0   
105     Chrome Dev  COMMUNICATION     4.4   63543.0   5000000.0   
109       GMX Mail  COMMUNICATION     4.3  258556.0  10000000.0   
118        GroupMe  COMMUNICATION     4.5  330761.0  10000000.0   

     Avg_Subjectivity  
55           0.675000  
58           0.655431  
105          0.520779  
109          0.540740  
118          0.518137  
